# ProFastNPV bug - simple example

LCOE is supposed to be the price where NPV = 0. Right now it's not.

In [1]:
import sys
sys.path.insert(0, "/Users/svijaysh/NPP+DCfork/H2Integrate")

import numpy as np
import yaml

from h2integrate.core.h2integrate_model import H2IntegrateModel
from h2integrate.core.file_utils import get_path
from h2integrate.finances.profast_base import ProFastBase
from h2integrate.finances.profast_npv import ProFastNPV

# grab the raw ProFAST object so we can call cash_flow() ourselves
saved_pf = {}
ProFastBase._orig = ProFastBase.populate_profast
def patched(self, inputs):
    pf = ProFastBase._orig(self, inputs)
    if isinstance(self, ProFastNPV):
        saved_pf["pf"] = pf
    return pf
ProFastBase.populate_profast = patched

In [2]:
example_dir = get_path("examples/32_nuclear_DC_case_1")
with open(example_dir / "plant_config.yaml") as f:
    plant_config = yaml.safe_load(f)

# no incentives, just plain cost
plant_config["finance_parameters"]["finance_groups"]["model_inputs"]["params"]["one_time_cap_inct"]["value"] = 0.0
plant_config["finance_parameters"]["finance_groups"]["npv"]["model_inputs"]["params"]["one_time_cap_inct"]["value"] = 0.0

model = H2IntegrateModel({
    "name": "simple_test",
    "system_summary": "simple test",
    "driver_config": str(example_dir / "driver_config.yaml"),
    "technology_config": str(example_dir / "tech_config.yaml"),
    "plant_config": plant_config,
})
model.setup()

capacity_kw = 2_400_000
model.prob.set_val("nuclear.system_capacity", capacity_kw, units="kW")
model.prob.set_val("grid.interconnection_size", capacity_kw, units="kW")
model.prob.set_val("nuclear_poi.interconnection_size", capacity_kw, units="kW")
model.prob.set_val("nuclear.electricity_command_value", np.full(8760, capacity_kw), units="kW")
model.prob.set_val("grid.electricity_command_value", np.full(8760, 500_000), units="kW")
model.run()

lcoe = float(model.prob.get_val("finance_subgroup_nuclear.LCOE", units="USD/(kW*h)")[0])
print("LCOE =", lcoe * 1000, "$/MWh")

/Users/svijaysh/Anaconda/anaconda3/envs/h2integrate/lib/python3.11/site-packages/openmdao/core/driver.py:879: OpenMDAOWarning:Driver: No matches for pattern '*resource_data' in recording_options['excludes'].


LCOE = 135.18487836273712 $/MWh


Here's the actual bug - the code zero-pads the construction years instead of using a real price:

In [3]:
pf = saved_pf["pf"]

non_op_years = 9  # construction period, from installation_time
plant_life = 80
price_array = np.full(plant_life, lcoe)

npv_zero_pad = pf.cash_flow(price=np.concatenate([np.zeros(non_op_years), price_array]))
npv_price_pad = pf.cash_flow(price=np.concatenate([np.full(non_op_years, lcoe), price_array]))

print("zero-padded (the bug):  ", npv_zero_pad / 1e9, "$B")
print("price-padded (the fix): ", npv_price_pad / 1e9, "$B")

zero-padded (the bug):   -0.9512624690330344 $B
price-padded (the fix):  9.186915121972562e-16 $B
